# 16 — Employee Intelligence Table
One row per employee: ID, Dept, Attrition_Prob, Risk, Role, Skill_Gap, Recommendation.

In [1]:

import pandas as pd
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

PROC = r'../data/processed'
MODELS = r'../models'

ea = pd.read_csv(f'{PROC}/employee_attrition_processed.csv')
feature_matrix = pd.read_csv(f'{PROC}/feature_matrix.csv')
emp_gap = pd.read_csv(f'{PROC}/employee_skill_gap_summary.csv')
recs_df = pd.read_csv(f'{PROC}/upskilling_recommendations.csv')
pipeline = joblib.load(f'{MODELS}/attrition_pipeline.joblib')
print(f"Loaded. ea: {ea.shape}, feature_matrix: {feature_matrix.shape}")


Loaded. ea: (500, 25), feature_matrix: (500, 45)


In [2]:

# ── Generate attrition probabilities for all employees ──
target = 'AttritionRisk_Label'
drop_cols = [c for c in [target, 'EmployeeID'] if c in feature_matrix.columns]
X_all = feature_matrix.drop(columns=drop_cols).astype(float)

attrition_probs = pipeline.predict_proba(X_all)[:, 1]
print(f"Attrition probability range: [{attrition_probs.min():.3f}, {attrition_probs.max():.3f}]")
print(f"Mean attrition probability: {attrition_probs.mean():.3f}")


Attrition probability range: [0.001, 0.999]
Mean attrition probability: 0.112


In [3]:

# ── Risk level assignment ──
# Thresholds chosen to reflect business cost asymmetry:
# HIGH >= 0.65: predict as at-risk with high confidence (act now)
# MEDIUM 0.40–0.65: watch closely, intervene proactively
# LOW < 0.40: stable, standard engagement
def assign_risk_level(prob):
    if prob >= 0.65:
        return 'HIGH'
    elif prob >= 0.40:
        return 'MEDIUM'
    else:
        return 'LOW'

risk_levels = [assign_risk_level(p) for p in attrition_probs]
print("Risk level distribution:")
from collections import Counter
print(Counter(risk_levels))


Risk level distribution:
Counter({'LOW': 445, 'HIGH': 55})


In [4]:

# ── Build master intelligence table ──
intelligence = ea[['EmployeeID','Name','Department','JobRole','Gender','Age',
                   'YearsAtCompany','MonthlySalary','PerformanceRating',
                   'WorkLifeBalanceScore','AttritionRisk']].copy()

intelligence['Attrition_Prob'] = np.round(attrition_probs, 4)
intelligence['Risk_Level'] = risk_levels

# Merge skill gap info
intelligence = intelligence.merge(
    emp_gap[['employee_id','gaps_count','weighted_gap_score','coverage_pct']].rename(
        columns={'employee_id':'EmployeeID'}),
    on='EmployeeID', how='left'
)

# Merge recommendations
intelligence = intelligence.merge(
    recs_df[['employee_id','top_skill_gap','priority','recommended_tools']].rename(
        columns={'employee_id':'EmployeeID','top_skill_gap':'Top_Skill_Gap',
                 'priority':'Rec_Priority','recommended_tools':'Recommended_Tools'}),
    on='EmployeeID', how='left'
)

# Fill nulls
intelligence['gaps_count'] = intelligence['gaps_count'].fillna(0).astype(int)
intelligence['weighted_gap_score'] = intelligence['weighted_gap_score'].fillna(0).round(2)
intelligence['coverage_pct'] = intelligence['coverage_pct'].fillna(0).round(3)
intelligence['Top_Skill_Gap'] = intelligence['Top_Skill_Gap'].fillna('No gaps identified')
intelligence['Rec_Priority'] = intelligence['Rec_Priority'].fillna('N/A')
intelligence['Recommended_Tools'] = intelligence['Recommended_Tools'].fillna('N/A')

print(f"Employee Intelligence Table: {intelligence.shape}")
print(f"Columns: {list(intelligence.columns)}")
print(f"\nSample (top 5 high risk):")
print(intelligence[intelligence['Risk_Level']=='HIGH'].head(5)[
    ['EmployeeID','Department','JobRole','Attrition_Prob','Risk_Level','gaps_count','Top_Skill_Gap']
].to_string(index=False))


Employee Intelligence Table: (500, 19)
Columns: ['EmployeeID', 'Name', 'Department', 'JobRole', 'Gender', 'Age', 'YearsAtCompany', 'MonthlySalary', 'PerformanceRating', 'WorkLifeBalanceScore', 'AttritionRisk', 'Attrition_Prob', 'Risk_Level', 'gaps_count', 'weighted_gap_score', 'coverage_pct', 'Top_Skill_Gap', 'Rec_Priority', 'Recommended_Tools']

Sample (top 5 high risk):
 EmployeeID Department         JobRole  Attrition_Prob Risk_Level  gaps_count         Top_Skill_Gap
          2      Sales Sales Executive          0.9922       HIGH           5     Critical Thinking
         10    Finance      Accountant          0.9907       HIGH           6 Reading Comprehension
         24         It          Tester          0.9981       HIGH           5      Active Listening
         46         It       Developer          0.9983       HIGH           5      Active Listening
         61         Hr      Hr Manager          0.9987       HIGH           6      Active Listening


In [5]:

# ── Summary statistics ──
print("=== Intelligence Table Summary ===")
print(f"Total employees: {len(intelligence)}")
print(f"\nRisk breakdown:")
print(intelligence['Risk_Level'].value_counts())
print(f"\nAvg attrition probability: {intelligence['Attrition_Prob'].mean():.3f}")
print(f"Avg skill gaps per employee: {intelligence['gaps_count'].mean():.1f}")
print(f"\nHigh-risk employees with high skill gaps:")
high_risk = intelligence[(intelligence['Risk_Level']=='HIGH') & (intelligence['gaps_count']>10)]
print(f"  Count: {len(high_risk)} (compound risk: attrition + skill deficit)")


=== Intelligence Table Summary ===
Total employees: 500

Risk breakdown:
Risk_Level
LOW     445
HIGH     55
Name: count, dtype: int64

Avg attrition probability: 0.112
Avg skill gaps per employee: 6.0

High-risk employees with high skill gaps:
  Count: 0 (compound risk: attrition + skill deficit)


In [6]:

# ── Save ──
intelligence.to_csv(f'{PROC}/employee_intelligence.csv', index=False)
print("Saved: employee_intelligence.csv")
print("\n=== NOTEBOOK 16 COMPLETE — Employee Intelligence Table Ready ===")
print("This table drives the FastAPI endpoints and Streamlit dashboard.")


Saved: employee_intelligence.csv

=== NOTEBOOK 16 COMPLETE — Employee Intelligence Table Ready ===
This table drives the FastAPI endpoints and Streamlit dashboard.


**Employee Intelligence Table complete.** One row per employee with attrition probability, risk level (HIGH/MEDIUM/LOW), skill gap count, weighted gap score, and top upskilling recommendation.